# 観測予測モデル（自己回帰・マスク予測・拡散）

この notebook では、観測を予測すると言っても『何を見てよいか』で問題の形が変わることを整理します。自己回帰・マスク予測・拡散は、同じ観測予測でも使ってよい情報の範囲が違います。


## 次を当てるにも、見てよい情報の範囲が違う

観測予測モデルと一口に言っても、自己回帰のように過去だけを使うもの、マスク予測のように前後から穴埋めするもの、拡散のようにノイズを削りながら戻すものでは、条件づけのしかたがまるで違います。最初は、『次の単語を当てる』『欠けた単語を埋める』『ぼやけた画像を少しずつ戻す』の三つは別の問題だ、と捉えると入りやすくなります。


この違いは、そのまま向いているタスクの違いになります。順番に先を予測したいのか、欠けた部分を埋めたいのか、ノイズに強く復元したいのかで、同じ『予測』でも設計を変える必要があります。


## 三つの予測は、何を入力にしてよいかで分かれる

自己回帰は過去だけ、マスク予測は周辺文脈、拡散はノイズを帯びた全体から少しずつ元へ戻します。ここでは 1 次元系列でその差を単純化し、使える情報の違いがどう誤差に現れるかを見ます。要するに、『先だけ知りたいのか』『抜けを埋めたいのか』『壊れたものを戻したいのか』で、最初から問題設定が違います。


## この notebook の見どころ

三つの MSE をただ比べるのではなく、どの条件でその予測が成り立っているかを意識しながら読みます。とくに `mask_idx` は『どこをわざと隠して穴埋めさせるか』を表し、反復デノイズは『1回で戻さず少しずつ戻す』流れを表しています。使える情報の範囲が大きく違うことに注目してください。


数値は同じ系列から計算していますが、タスク設定まで完全に同一ではありません。だから『どれが一番強いか』ではなく、『何を入力にしてよい設計か』として比較してください。同じテストで速さを比べているのではなく、別々の仕事のやり方を並べている、という意識が大事です。


## 反復計算が増える意味

拡散型は一回で答えを出す代わりに、少しずつ戻る方向を積み上げます。計算は重くなりますが、ノイズに対して粘り強い復元がしやすく、その性格がここでも少し見えます。これは『一発で正解を書く』より『何度も手直ししながら元へ近づける』やり方だと思うと分かりやすくなります。


## 読み方の軸

逐次予測、欠損補完、ノイズ除去を同じ問題だと思わないことが重要です。入力条件の違いがそのままモデルの役割の違いになります。読み終わったときに、『未来は見てよいか』『前後は見てよいか』『壊れた全体を見てよいか』の違いを言えれば十分です。


## 過去だけを見て次を当てる

まずは自己回帰の流儀で、順番に次の値を予測する形を確認します。ここでは未来や周辺は見ず、使ってよいのは過去だけです。言い換えると、『その時点までに机の上へ出ている情報だけで次を書く』設定です。


In [ ]:
import numpy as np
np.random.seed(1)

T = 80
t = np.arange(T)
series = np.sin(t / 6.0) + 0.1 * np.random.randn(T)


## 周辺情報やデノイズで埋める

続いて、穴埋め型と拡散型の復元を並べ、使える文脈の違いがどこに効くかを見ます。自己回帰と違って、ここでは前後や全体を参照してよい設定になります。穴埋め型は『空欄の前後を読む』、拡散型は『壊れた全体を見て何度も手直しする』と考えると整理しやすくなります。


In [ ]:
# 1) 自己回帰: x_t = a*x_{t-1} + b
x = series[:-1]
y = series[1:]
a = np.dot(x, y) / np.dot(x, x)
b = y.mean() - a * x.mean()
ar_pred = a * x + b
ar_mse = np.mean((ar_pred - y) ** 2)

# 2) マスク予測: 中央点を前後平均で補完
mask_idx = np.arange(1, T - 1, 5)
masked = series.copy()
masked[mask_idx] = np.nan
filled = masked.copy()
for i in mask_idx:
    filled[i] = 0.5 * (masked[i - 1] + masked[i + 1])
mask_mse = np.mean((filled[mask_idx] - series[mask_idx]) ** 2)

# 3) 拡散風予測: ノイズ付加 -> 逐次平滑で復元
noisy = series + 0.35 * np.random.randn(T)
den = noisy.copy()
for _ in range(20):
    den[1:-1] = 0.25 * den[:-2] + 0.5 * den[1:-1] + 0.25 * den[2:]
diff_mse = np.mean((den - series) ** 2)

print('AR MSE         =', round(ar_mse, 6))
print('Mask-fill MSE  =', round(mask_mse, 6))
print('Diffusion-like =', round(diff_mse, 6))


この notebook の要点は、三方式を順位づけすることではありません。何を条件として観測を予測するのか、その違いがモデル選択そのものを決めると見ることです。実務でも、次生成・欠損補完・ノイズ除去を混同せず、『いま何を見てよい問題なのか』からモデルを選ぶのが出発点になります。
